In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (classification_report, confusion_matrix,
    roc_curve, auc, f1_score, precision_score, recall_score,
    precision_recall_curve)
from sklearn.utils.class_weight import compute_sample_weight
from scipy.sparse import hstack, csr_matrix
import warnings; warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("C:\\Users\\AVI SHARMA\\Documents\\fake real job\\fake_job_postings.csv")
df.head(5)

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


Combine text columns

In [4]:
df['text'] = (df['title'].fillna('') + ' ' +
              df['description'].fillna('') + ' ' +
              df['requirements'].fillna('') + ' ' +
              df['company_profile'].fillna(''))


Engineer numeric features

In [5]:
df['desc_len'] = df['description'].fillna('').apply(len)
df['req_len']  = df['requirements'].fillna('').apply(len)
df['text_len'] = df['text'].apply(len)

y = df['fraudulent'].values

Class Distribution

In [6]:
class_counts = df['fraudulent'].value_counts().reset_index()
class_counts.columns = ['Class', 'Count']
class_counts['Label'] = class_counts['Class'].map({0: 'Real (0)', 1: 'Fake (1)'})

fig = px.bar(
    class_counts, x='Label', y='Count',
    color='Label',
    color_discrete_map={'Real (0)': '#2196F3', 'Fake (1)': '#F44336'},
    title='Figure 1: Class Distribution — Real vs Fake Job Postings',
    text='Count'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, xaxis_title='Class', yaxis_title='Count')
fig.show()

Missing Value Analysis

In [7]:
missing = df.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False).reset_index()
missing.columns = ['Column', 'Missing %']

fig = px.bar(
    missing, x='Missing %', y='Column',
    orientation='h',
    title='Figure 2: Missing Value Analysis by Column (%)',
    color='Missing %',
    color_continuous_scale='Reds',
    text=missing['Missing %'].round(1).astype(str) + '%'
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

Binary Feature Analysis

In [8]:
binary_cols = ['telecommuting', 'has_company_logo', 'has_questions']
fig = make_subplots(rows=1, cols=3, subplot_titles=binary_cols)

for i, col in enumerate(binary_cols):
    grouped = df.groupby([col, 'fraudulent']).size().reset_index(name='count')
    grouped['fraudulent_label'] = grouped['fraudulent'].map({0: 'Real', 1: 'Fake'})
    grouped[col] = grouped[col].astype(str)

    for label, color in zip(['Real', 'Fake'], ['#2196F3', '#F44336']):
        subset = grouped[grouped['fraudulent_label'] == label]
        fig.add_trace(
            go.Bar(x=subset[col], y=subset['count'],
                   name=label, marker_color=color,
                   showlegend=(i == 0)),
            row=1, col=i+1
        )

fig.update_layout(
    title_text='Figure 3: Binary Feature Analysis — Telecommuting, Logo, Questions',
    barmode='group', height=400
)
fig.show()

Employment Type vs Fraud Rate


In [9]:
emp_fraud = df.groupby('employment_type')['fraudulent'].mean().reset_index()
emp_fraud.columns = ['Employment Type', 'Fraud Rate']
emp_fraud = emp_fraud.sort_values('Fraud Rate', ascending=False).dropna()

fig = px.bar(
    emp_fraud, x='Employment Type', y='Fraud Rate',
    title='Figure 4: Employment Type vs Fraud Rate',
    color='Fraud Rate', color_continuous_scale='OrRd',
    text=emp_fraud['Fraud Rate'].round(3).astype(str)
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='Employment Type', yaxis_title='Fraud Rate')
fig.show()

Top Industries by Fraud Rate

In [10]:
ind_fraud = df.groupby('industry')['fraudulent'].mean().reset_index()
ind_fraud.columns = ['Industry', 'Fraud Rate']
ind_fraud = ind_fraud.sort_values('Fraud Rate', ascending=False).head(15).dropna()

fig = px.bar(
    ind_fraud, x='Fraud Rate', y='Industry',
    orientation='h',
    title='Figure 5: Top 15 Industries by Fraud Rate',
    color='Fraud Rate', color_continuous_scale='Reds',
    text=ind_fraud['Fraud Rate'].round(3).astype(str)
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

Text Length Distribution — Real vs Fake

In [11]:
df['label'] = df['fraudulent'].map({0: 'Real', 1: 'Fake'})

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Description Length', 'Requirements Length', 'Total Text Length'])

cols_to_plot = ['desc_len', 'req_len', 'text_len']
colors = {'Real': '#2196F3', 'Fake': '#F44336'}

for i, col in enumerate(cols_to_plot):
    for label, color in colors.items():
        subset = df[df['label'] == label][col]
        fig.add_trace(
            go.Histogram(x=subset, name=label,
                         marker_color=color, opacity=0.6,
                         showlegend=(i == 0)),
            row=1, col=i+1
        )

fig.update_layout(
    title_text='Figure 6: Text Length Distribution — Real vs Fake',
    barmode='overlay', height=400
)
fig.show()

TF-IDF Vectorization & Feature Matrix

In [12]:
tfidf = TfidfVectorizer(
    max_features=4000, ngram_range=(1, 2),
    sublinear_tf=True, stop_words='english', min_df=3
)
X_tfidf = tfidf.fit_transform(df['text'])
X_num = df[['telecommuting', 'has_company_logo',
            'has_questions', 'desc_len', 'req_len', 'text_len']
          ].values.astype(float)
X_all = hstack([X_tfidf, csr_matrix(X_num)])

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=42, stratify=y
)

 Model Training

In [13]:
models = {
    'Logistic Regression': LogisticRegression(
        C=1.0, class_weight='balanced',
        max_iter=300, solver='saga'),
    'Random Forest': RandomForestClassifier(
        n_estimators=80, max_depth=12,
        class_weight='balanced', n_jobs=2),
    'Linear SVM': CalibratedClassifierCV(
        LinearSVC(C=0.5, class_weight='balanced', max_iter=500)),
    'MLP Deep NN': MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),
        activation='relu', solver='adam', alpha=0.001,
        max_iter=100, early_stopping=True)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'pred': y_pred,
        'prob': y_prob,
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
    }
    print(name)
    print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))

Logistic Regression
              precision    recall  f1-score   support

        Real       0.98      0.72      0.83      3403
        Fake       0.11      0.66      0.19       173

    accuracy                           0.72      3576
   macro avg       0.54      0.69      0.51      3576
weighted avg       0.93      0.72      0.80      3576

Random Forest
              precision    recall  f1-score   support

        Real       0.99      0.96      0.97      3403
        Fake       0.50      0.79      0.61       173

    accuracy                           0.95      3576
   macro avg       0.75      0.87      0.79      3576
weighted avg       0.97      0.95      0.96      3576

Linear SVM
              precision    recall  f1-score   support

        Real       0.99      1.00      0.99      3403
        Fake       0.91      0.79      0.84       173

    accuracy                           0.99      3576
   macro avg       0.95      0.89      0.92      3576
weighted avg       0.99      

ROC Curves

In [14]:
fig = go.Figure()

for name, model in models.items():
    y_prob = results[name]['prob']
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f'{name} (AUC={roc_auc:.3f})'
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    line=dict(dash='dash', color='gray'),
    name='Random Classifier'
))

fig.update_layout(
    title='Figure 8: ROC Curves — All Models',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    legend=dict(x=0.6, y=0.1)
)
fig.show()

Confusion Matrices (All 4 Models)

In [15]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=list(models.keys())
)

positions = [(1,1), (1,2), (2,1), (2,2)]

for (row, col), name in zip(positions, models.keys()):
    cm = confusion_matrix(y_test, results[name]['pred'])
    fig.add_trace(
        go.Heatmap(
            z=cm, x=['Real', 'Fake'], y=['Real', 'Fake'],
            colorscale='Blues',
            text=cm, texttemplate='%{text}',
            showscale=False
        ),
        row=row, col=col
    )

fig.update_layout(
    title_text='Figure 9: Confusion Matrices — All 4 Models',
    height=600
)
fig.show()

Precision-Recall Curves

In [16]:
fig = go.Figure()

for name, model in models.items():
    y_prob = results[name]['prob']
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    fig.add_trace(go.Scatter(
        x=rec, y=prec, mode='lines',
        name=name
    ))

fig.update_layout(
    title='Figure 10: Precision-Recall Curves — All Models',
    xaxis_title='Recall',
    yaxis_title='Precision'
)
fig.show()

Top Discriminative Words — Fake vs Real

In [18]:
lr_model = models['Logistic Regression']
feature_names = tfidf.get_feature_names_out()
tfidf_coefficients = lr_model.coef_[0][:len(feature_names)]
top_n = 15
top_fake_idx  = np.argsort(tfidf_coefficients)[-top_n:][::-1]
top_real_idx  = np.argsort(tfidf_coefficients)[:top_n]

words  = (list(feature_names[top_fake_idx]) +
          list(feature_names[top_real_idx]))
scores = (list(tfidf_coefficients[top_fake_idx]) +
          list(tfidf_coefficients[top_real_idx]))
labels = (['Fake Indicator'] * top_n +
          ['Real Indicator'] * top_n)

feat_df = pd.DataFrame({'Word': words, 'Coefficient': scores, 'Type': labels})

fig = px.bar(
    feat_df, x='Coefficient', y='Word',
    color='Type', orientation='h',
    color_discrete_map={'Fake Indicator': '#F44336', 'Real Indicator': '#2196F3'},
    title='Figure 11: Top Discriminative Words — Fake vs Real Job Indicators'
)
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=700
)
fig.show()